# Process PB2 DMS data from Soh et al.

Import Python modules

In [ ]:
import os
import pandas as pd
from collections import defaultdict
from Bio import SeqIO
import subprocess

Align the unmutated DMS sequence and the sequence used as reference for tree building. See if there are any gaps.

In [ ]:
# Read in the sequence of the reference NP used to build the tree
# TODO: update path using papermill
ref_fasta = '../../flu-usher/results_260226/PB2/all/curated_reference.fasta'
ref_seq = SeqIO.read(ref_fasta, 'fasta').seq
ref_aa_seq = str(ref_seq.translate())
ref_aa_seq = ref_aa_seq.replace('*', '')
print('reference sequence length:', len(ref_aa_seq))

# Read in the DMS data and get the sequence of the unmutated protein
pb2_dms_data = (
    pd.read_csv('../data/dms_data/Soh_PB2/elife-45079-fig2-data1-v1.csv')
    .rename(columns={
        'log2effectA549' : 'dms_effect', 
        'wildtype' : 'wt_aa', 
        'mutation' : 'mut_aa'
    })
)
aa_seq = ''.join(list(
    pb2_dms_data
    .sort_values('site')
    .drop_duplicates('site')
    ['wt_aa']
))
print('DMS sequence length:', len(aa_seq))

# Save sequences to a FASTA file for alignment
output_dir = '../results/dms_data/Soh_PB2/'
if not os.path.isdir(output_dir):
    os.makedirs(output_dir)
unaligned_fasta = os.path.join(output_dir, 'unaligned.fasta')
if not os.path.isfile(unaligned_fasta):
    with open(unaligned_fasta, 'w') as f:
        f.write('>reference\n')
        f.write(ref_aa_seq + '\n')
        f.write('>dms\n')
        f.write(aa_seq + '\n')

# Align the sequences using MUSCLE
aligned_fasta = f'{output_dir}/aligned.fasta'
if not os.path.isfile(aligned_fasta):
    cmd = ['muscle', '-align', unaligned_fasta, '-output', aligned_fasta]
    result = subprocess.run(cmd, capture_output=True, text=True, check=True)

# Read in the aligned sequences
seqs_dict = defaultdict(list)
aligned_records = list(SeqIO.parse(aligned_fasta, 'fasta'))
for record in aligned_records:
    seqs_dict[record.id] = str(record.seq)

aligned_ref_seq = seqs_dict['reference']
aligned_dms_seq = seqs_dict['dms']

if '-' not in aligned_dms_seq and '-' not in aligned_ref_seq:
    print('No gaps in the alignment')

# Compute the percent identity
id = sum([a == b for (a, b) in zip(aligned_ref_seq, aligned_dms_seq)]) / len(aligned_ref_seq)
print('Percent identity:', id)